# File Knowledge Source: Upload files and query with agentic retrieval

This notebook demonstrates **File Knowledge Sources** in Azure AI Search. Unlike indexed knowledge sources (which require pre-built search indexes), file knowledge sources let you upload raw files directly — the service handles chunking, embedding, and indexing automatically.

You'll:
- Create a File Knowledge Source with an embedding model for ingestion
- Upload documents directly to the knowledge source
- Build a knowledge base that references the file knowledge source
- Query the knowledge base and get citation-backed answers

<details>
<summary><b>🗂️ What is a File Knowledge Source?</b></summary>

A **File Knowledge Source** accepts raw document uploads (PDF, Markdown, text, etc.) and automatically processes them:
- Extracts text content from uploaded files
- Chunks the content into searchable segments
- Generates vector embeddings using your configured embedding model
- Makes everything available for agentic retrieval queries

This is ideal when you want to quickly add documents to a knowledge base without setting up a full indexing pipeline.
</details>

## Step 1: Load environment variables

Load the configuration for your Azure resources.

> **ℹ️ Note**
>
> - The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
>
> - You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

# Azure AI Search configuration
endpoint = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
credential = AzureKeyCredential(os.environ["AZURE_SEARCH_ADMIN_KEY"])

# Azure OpenAI configuration
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
azure_openai_key = os.environ["AZURE_OPENAI_KEY"]
azure_openai_chatgpt_deployment = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
azure_openai_chatgpt_model_name = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
azure_openai_embedding_deployment = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
azure_openai_embedding_model = "text-embedding-3-large"

# Knowledge base and source names
file_knowledge_source_name = "file-docs-knowledge-source"
knowledge_base_name = "file-docs-knowledge-base"

print("Environment variables loaded")

## Step 2: Create a File Knowledge Source

A **File Knowledge Source** is different from an indexed knowledge source — instead of pointing to a pre-built search index, it accepts raw file uploads and handles ingestion automatically.

The key configuration is the `ingestion_parameters`, which specifies:
- **Embedding model**: An Azure OpenAI embedding model that generates embeddings for the uploaded content
- **Content extraction mode**: How to extract text and media from documents. For now, only "minimal" is supported for file knowledge sources.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    FileKnowledgeSource,
    FileKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

index_client = SearchIndexClient(endpoint=endpoint, credential=credential)

file_ks = FileKnowledgeSource(
    name=file_knowledge_source_name,
    description="Uploaded HR and company documents for Zava",
    file_parameters=FileKnowledgeSourceParameters(
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=azure_openai_endpoint,
                    deployment_name=azure_openai_embedding_deployment,
                    model_name=azure_openai_embedding_model,
                    api_key=azure_openai_key,
                )
            ),
            content_extraction_mode="minimal",
        )
    ),
)

index_client.create_or_update_knowledge_source(file_ks)
print(f"File knowledge source '{file_knowledge_source_name}' created or updated successfully.")

## Step 3: Upload files to the knowledge source

Now upload documents directly to the file knowledge source. The service will automatically:
1. Extract text from the uploaded files
2. Chunk the content into searchable segments
3. Generate vector embeddings

We'll upload the Zava Company Overview document and the health benefits PDFs from the lab's data directory.

In [ ]:
from pathlib import Path

# Paths to documents to upload
data_dir = Path("..") / "data" / "ai-search-data"
files_to_upload = [
    data_dir / "filesource" / "MSFT_cloud_architecture_zava.pdf"
]

uploaded_files = []
for file_path in files_to_upload:
    with open(file_path, "rb") as f:
        file_content = f.read()
    uploaded = index_client.upload_knowledge_source_file(
        file_knowledge_source_name, file_content
    )
    uploaded_files.append(uploaded)
    print(f"Uploaded: {file_path.name} (file_id: {uploaded.file_id}, {uploaded.file_size_bytes} bytes)")

print(f"\nTotal files uploaded: {len(uploaded_files)}")

## Step 4: Verify uploaded files

List all files in the knowledge source to confirm the uploads were successful.

In [ ]:
files = list(index_client.list_knowledge_source_files(file_knowledge_source_name))
print(f"Files in '{file_knowledge_source_name}':\n")
for file in files:
    print(f"  {file.file_name}")
    print(f"    ID: {file.file_id}")
    print(f"    Size: {file.file_size_bytes} bytes")
    print(f"    Created: {file.created_at}")
    print(f"    Last updated: {file.last_updated_at}")
    if file.error_message:
        print(f"    ERROR: {file.error_message}")
    print()
print(f"Total: {len(files)} file(s)")

## Step 5: Create a knowledge base

Now create a knowledge base that references the file knowledge source. The knowledge base combines:
1. Your file knowledge source (the uploaded documents)
2. An Azure OpenAI chat model for reasoning and answer synthesis

With `output_mode=ANSWER_SYNTHESIS`, the knowledge base uses the chat model to generate natural language answers from the retrieved document chunks.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalOutputMode,
)

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=azure_openai_endpoint,
    deployment_name=azure_openai_chatgpt_deployment,
    model_name=azure_openai_chatgpt_model_name,
    api_key=azure_openai_key,
)

knowledge_base = KnowledgeBase(
    name=knowledge_base_name,
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name=file_knowledge_source_name),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{knowledge_base_name}' created or updated successfully.")

## Step 6: Query the knowledge base

Ask a question that spans the uploaded documents. The knowledge base will:
1. Analyze your query
2. Search across the uploaded file content
3. Synthesize a natural language answer with citations

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=endpoint, knowledge_base_name=knowledge_base_name, credential=credential
)

file_ks_params = FileKnowledgeSourceParams(
    knowledge_source_name=file_knowledge_source_name,
    include_references=True,
    include_reference_source_data=True,
)

req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(
                    text="What are the levels of Zava data sensitivity classification?"
                )
            ],
        )
    ],
    knowledge_source_params=[file_ks_params],
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=req)
display(Markdown(result.response[0].content[0].text))

activity_by_id = {activity.id: getattr(activity, "knowledge_source_name", None) for activity in (result.activity or [])}
references_json = []

for reference in result.references or []:
    reference_json = reference.as_dict()
    reference_json["knowledgeSourceName"] = activity_by_id.get(reference.activity_source, "unknown")
    references_json.append(reference_json)

print(json.dumps(references_json, indent=2))

## Step 7: Review activity log

The activity log shows how the agentic retrieval processed your query — what subqueries were generated and how the file knowledge source was searched.

In [ ]:
import json

activity_types = [a.type for a in result.activity]
print("Activity Log Steps")
for i, activity_type in enumerate(activity_types):
    print(f"  {i}: {activity_type}")

print("\nActivity Details")
print(json.dumps([a.as_dict() for a in result.activity], indent=2))

## Summary

You've completed the File Knowledge Source notebook! You uploaded raw documents directly to a file knowledge source, built a knowledge base around it, and queried with agentic retrieval.

**Key concepts:**
- **File Knowledge Sources** accept raw uploads — no pre-built index required
- Ingestion handles chunking and embedding automatically using your configured embedding model
- `FileKnowledgeSourceParams` is used at query time to include file sources in retrieval
- The knowledge base synthesizes answers across all uploaded file content
- Activity logs show how queries were decomposed and searched

**When to use File Knowledge Sources vs. Indexed Knowledge Sources:**
| | File Knowledge Source | Indexed Knowledge Source |
|---|---|---|
| Setup | Upload files directly | Pre-build a search index |
| Best for | Quick prototyping, small doc sets | Production workloads, large datasets |
| Control | Automatic chunking/embedding | Full control over schema and processing |